# CS604 数据：一次性下载到 Google Drive

把 Hugging Face 上课程数据集仓库里的**全部文件**下载到你的 Google Drive（默认放在 `我的云端硬盘/cs604/data/`）。目前包括 train/val 的压缩包和图片列表、`README.md`、`metadata.json`。之后训练 notebook 会直接从 Drive 解压，不用每次都从 HF 下载。

**使用方法**
1. **运行时选 CPU 就够了**（`代码执行程序 → 更改运行时类型 → CPU`），不需要 GPU，不浪费计算单元。
2. `代码执行程序 → 全部运行`，按提示授权 Google Drive。
3. **保持这个页面开着**，直到最后一个单元打印 `Done`。整个过程大约 20–60 分钟，取决于当时的网速。

**注意**
- 需要约 **45GB** 的 Drive 剩余空间（`train.tar.gz` 40.1GB + `val.tar.gz` 4.4GB）。免费版 Drive 只有 15GB，需要先扩容。
- 数据从 HF 边下载边写进 Drive，不在 Colab 本地多存一份。
- 中途断线的话重新「全部运行」即可：已完成的文件会跳过，没下完的从断点继续。
- Phase 2 发布测试集后，再「全部运行」一次即可：会自动补上仓库里新增的文件（如 `test.tar.gz`、`test.txt`），已下载的会跳过。

In [ ]:
HF_REPO = "doem1997/cs604-course-data"
DRIVE_DATA_DIR = "/content/drive/MyDrive/cs604/data"  # 训练 notebook 默认从这里读取
SKIP_SPACE_CHECK = False     # 确认 Drive 空间足够、但检查误报时改成 True

In [ ]:
import os, time, shutil
import requests
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_url
from huggingface_hub.hf_api import RepoFile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

## 检查要下载的文件和 Drive 空间

In [ ]:
# 列出仓库里的全部文件，按大小从小到大下载（train.tar.gz 最后）
repo_files = [f for f in HfApi().list_repo_tree(HF_REPO, repo_type="dataset", recursive=True) if isinstance(f, RepoFile)]
sizes = {f.path: f.size for f in repo_files}
names = sorted(sizes, key=sizes.get)


def drive_path(name):
    return os.path.join(DRIVE_DATA_DIR, name)


def in_drive(name):
    return os.path.exists(drive_path(name)) and os.path.getsize(drive_path(name)) == sizes[name]


def partial_bytes(name):
    part = drive_path(name) + ".part"
    return os.path.getsize(part) if os.path.exists(part) else 0


todo = [n for n in names if not in_drive(n)]
need = sum(sizes[n] - partial_bytes(n) for n in todo)
free = shutil.disk_usage(DRIVE_DATA_DIR).free
for n in names:
    status = "already in Drive" if n not in todo else f"to download ({partial_bytes(n) / 1e9:.1f} GB done)"
    print(f"{n:16s} {sizes[n] / 1e9:7.2f} GB   {status}")
print(f"\nNeed {need / 1e9:.1f} GB more; Drive reports {free / 1e9:.1f} GB free")
if need > free and not SKIP_SPACE_CHECK:
    raise RuntimeError("Drive 剩余空间不够：请先清理或扩容 Google Drive。"
                       "如果确认空间足够、只是检查结果不准，把 SKIP_SPACE_CHECK 改成 True 再运行")

## 下载到 Drive（断点续传）

In [ ]:
def download(name, retries=10):
    """从 HF 流式下载到 Drive 里的 .part 文件，中断后用 HTTP Range 从断点继续，完成后改名"""
    size, part = sizes[name], drive_path(name) + ".part"
    os.makedirs(os.path.dirname(part), exist_ok=True)
    url = hf_hub_url(HF_REPO, name, repo_type="dataset")
    for attempt in range(1, retries + 1):
        done = partial_bytes(name)
        if done > size:  # 残留文件异常，从头下载
            os.remove(part)
            done = 0
        if done < size:
            headers = {"Accept-Encoding": "identity"}
            if done:
                headers["Range"] = f"bytes={done}-"
            try:
                with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    if done and r.status_code != 206:  # 服务器没有按断点返回，只能从头下载
                        done = 0
                    with open(part, "ab" if done else "wb") as f, \
                            tqdm(total=size, initial=done, unit="B", unit_scale=True, desc=name) as bar:
                        for chunk in r.iter_content(chunk_size=8 << 20):
                            f.write(chunk)
                            bar.update(len(chunk))
            except (requests.RequestException, OSError) as e:
                print(f"{name}: attempt {attempt} interrupted ({e!r}); resuming in 10 s")
                time.sleep(10)
                if isinstance(e, OSError) and not os.path.isdir(DRIVE_DATA_DIR):
                    drive.mount("/content/drive", force_remount=True)  # Drive 挂载掉线时重新挂载
                continue
        if partial_bytes(name) == size:
            os.replace(part, drive_path(name))
            return
    raise RuntimeError(f"{name}: still incomplete after {retries} attempts; re-run this cell to resume")


t0 = time.time()
for name in todo:
    download(name)
print(f"All files written to {DRIVE_DATA_DIR} in {(time.time() - t0) / 60:.1f} min")

## 等 Drive 上传完成

写进 `/content/drive` 的数据会先缓存在 Colab 本地，再在后台上传。下面这一步会等所有数据真正传到 Google Drive 之后才结束，**在它打印 `Done` 之前不要关闭页面或断开运行时**。

In [ ]:
bad = [n for n in names if not in_drive(n)]
assert not bad, f"incomplete files: {bad}; re-run the download cell"
print("Waiting for Google Drive to finish uploading (can take a while for 40+ GB) ...")
t0 = time.time()
drive.flush_and_unmount()
print(f"Done in {(time.time() - t0) / 60:.1f} min. Files in Google Drive: {DRIVE_DATA_DIR}")
for n in names:
    print(f"  {n:16s} {sizes[n] / 1e9:7.2f} GB")

## 下一步

训练 notebook（`cs604_vit_adaptation.ipynb`）的 `TAR_SOURCE_DIR` 默认就是 `/content/drive/MyDrive/cs604/data`。只要这里有数据，训练时就会直接从 Drive 解压到 Colab 本地磁盘，不再从 HF 下载。

每个新会话仍然需要解压一次（几分钟），因为训练必须从 Colab 本地磁盘读图：直接从 Drive 逐个读取 45 万张小图会非常慢。